# Falcon FLM — Google Colab backend\n\nBu notebook GitHub'daki **aynı FLM eğitim motorunu** Colab GPU üzerinde çalıştırır. Checkpoint ve metrikler Google Drive'a yazılır. Hazır Llama/Qwen modeli veya tokenizer kullanılmaz.

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\n

In [ ]:
import os, pathlib, subprocess, sys\n\nREPO_URL = 'https://github.com/hanefimert2016-oss/falcon-kaggle-trainer.git'\nROOT = pathlib.Path('/content/falcon-kaggle-trainer')\n\nif (ROOT / '.git').exists():\n    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', 'main'], check=True)\n    subprocess.run(['git', '-C', str(ROOT), 'reset', '--hard', 'origin/main'], check=True)\nelse:\n    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(ROOT)], check=True)\n\nos.chdir(ROOT)\nsys.path.insert(0, str(ROOT))\nprint('repo=', ROOT)\n

In [ ]:
from scripts.colab_setup import prepare\n\nruntime = prepare('/content/drive/MyDrive/FalconFLM')\nruntime\n

## Veri\nGerçek eğitim için `.txt` dosyalarını aşağıdaki Drive klasörüne koy:\n\n`MyDrive/FalconFLM/data/`\n\nKlasör boşsa eğitim hattını test etmek için küçük dahili fallback veri kullanılır.

In [ ]:
# İlk deneme için güvenli kısa ayarlar. İstersen daha sonra büyütebiliriz.\nos.environ['FLM_STEPS'] = '50'\nos.environ['FLM_SEQ_LEN'] = '256'\nos.environ['FLM_LAYERS'] = '8'\nos.environ['FLM_HEADS'] = '8'\nos.environ['FLM_EMBD'] = '512'\nos.environ['FLM_BATCH'] = '16'\nos.environ['FLM_GRAD_ACCUM'] = '4'\nos.environ['FLM_EVAL_INTERVAL'] = '100'\nos.environ['FLM_EVAL_ITERS'] = '10'\n\nprint('data=', os.environ['FLM_DATA_DIR'])\nprint('output=', os.environ['FLM_OUTPUT_DIR'])\n

In [ ]:
subprocess.run([sys.executable, 'kaggle/train_flm.py'], check=True, env=os.environ.copy())\n

In [ ]:
import json\nout = pathlib.Path(os.environ['FLM_OUTPUT_DIR'])\nprint('files:', [p.name for p in out.iterdir()])\nprint(json.dumps(json.loads((out / 'metrics.json').read_text()), indent=2))\nprint('checkpoint:', out / 'checkpoint.pt')\n